# Malware Classification with Extra Tree Classifier and Logistic Regression

This notebook separates legitimate and malware samples and trains models to classify malware types.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Load the dataset
df = pd.read_csv('dataset/merged_dataset.csv', sep='|')
print(f"Dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
df.head()

In [ ]:
# Check data distribution
print("Legitimate vs Malware distribution:")
print(df['legitimate'].value_counts())
print("\nMalware types distribution:")
print(df['MalwareType'].value_counts())

In [ ]:
# Separate legitimate and malware samples
legitimate_samples = df[df['legitimate'] == 0].copy()
malware_samples = df[df['legitimate'] == 1].copy()

print(f"Legitimate samples: {len(legitimate_samples)}")
print(f"Malware samples: {len(malware_samples)}")

# Save separated datasets
legitimate_samples.to_csv('dataset/legitimate_samples.csv', index=False)
malware_samples.to_csv('dataset/malware_samples.csv', index=False)

print("\nSeparated datasets saved!")

In [ ]:
# Focus on malware classification - remove legitimate samples
malware_df = malware_samples.copy()

# Remove non-feature columns
feature_columns = malware_df.columns.drop(['Name', 'md5', 'legitimate', 'MalwareType'])
X = malware_df[feature_columns]
y = malware_df['MalwareType']

print(f"Features shape: {X.shape}")
print(f"Target classes: {y.unique()}")
print(f"Class distribution:\n{y.value_counts()}")

In [ ]:
# Handle missing values and data preprocessing
print(f"Missing values per column:\n{X.isnull().sum().sum()}")

# Fill missing values with median
X = X.fillna(X.median())

# Encode target labels
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print(f"Encoded classes: {label_encoder.classes_}")

In [ ]:
# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")

In [ ]:
# Scale features for Logistic Regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Train Extra Trees Classifier
print("Training Extra Trees Classifier...")
extra_trees = ExtraTreesClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)
extra_trees.fit(X_train, y_train)

# Predictions
et_pred = extra_trees.predict(X_test)
et_accuracy = accuracy_score(y_test, et_pred)

print(f"Extra Trees Accuracy: {et_accuracy:.4f}")

In [ ]:
# Train Logistic Regression
print("Training Logistic Regression...")
logistic_reg = LogisticRegression(
    random_state=42,
    max_iter=1000,
    multi_class='ovr'
)
logistic_reg.fit(X_train_scaled, y_train)

# Predictions
lr_pred = logistic_reg.predict(X_test_scaled)
lr_accuracy = accuracy_score(y_test, lr_pred)

print(f"Logistic Regression Accuracy: {lr_accuracy:.4f}")

In [ ]:
# Detailed evaluation for Extra Trees
print("=== Extra Trees Classifier Results ===")
print(classification_report(y_test, et_pred, target_names=label_encoder.classes_))

# Confusion Matrix
plt.figure(figsize=(10, 8))
cm_et = confusion_matrix(y_test, et_pred)
sns.heatmap(cm_et, annot=True, fmt='d', cmap='Blues', 
            xticklabels=label_encoder.classes_, 
            yticklabels=label_encoder.classes_)
plt.title('Extra Trees - Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

In [ ]:
# Detailed evaluation for Logistic Regression
print("=== Logistic Regression Results ===")
print(classification_report(y_test, lr_pred, target_names=label_encoder.classes_))

# Confusion Matrix
plt.figure(figsize=(10, 8))
cm_lr = confusion_matrix(y_test, lr_pred)
sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Greens', 
            xticklabels=label_encoder.classes_, 
            yticklabels=label_encoder.classes_)
plt.title('Logistic Regression - Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

In [ ]:
# Feature importance for Extra Trees
feature_importance = pd.DataFrame({
    'feature': feature_columns,
    'importance': extra_trees.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(12, 8))
sns.barplot(data=feature_importance.head(20), x='importance', y='feature')
plt.title('Top 20 Feature Importances - Extra Trees')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

print("Top 10 most important features:")
print(feature_importance.head(10))

In [ ]:
# Model comparison
results = pd.DataFrame({
    'Model': ['Extra Trees', 'Logistic Regression'],
    'Accuracy': [et_accuracy, lr_accuracy]
})

plt.figure(figsize=(8, 6))
sns.barplot(data=results, x='Model', y='Accuracy')
plt.title('Model Comparison - Accuracy')
plt.ylim(0, 1)
for i, v in enumerate(results['Accuracy']):
    plt.text(i, v + 0.01, f'{v:.4f}', ha='center')
plt.show()

print("\nModel Performance Summary:")
print(results)

In [ ]:
# Save the trained models
import joblib

# Save models
joblib.dump(extra_trees, 'models/extra_trees_malware_classifier.pkl')
joblib.dump(logistic_reg, 'models/logistic_regression_malware_classifier.pkl')
joblib.dump(scaler, 'models/scaler_malware.pkl')
joblib.dump(label_encoder, 'models/label_encoder_malware.pkl')

print("Models saved successfully!")
print(f"Extra Trees Accuracy: {et_accuracy:.4f}")
print(f"Logistic Regression Accuracy: {lr_accuracy:.4f}")